# Chapter 6 — Building Your First Deep Neural Network: Introduction to Backpropagation

**Mata Kuliah:** Deep Learning
**Buku:** Grokking Deep Learning (Andrew W. Trask, Manning)
**Chapter 6:** Building Your First Deep Neural Network: Introduction to Backpropagation

Notebook ini berisi reproduksi kode dari buku (diambil dari repository
resmi penulis) beserta penjelasan konsep dan rangkuman chapter.

## Pendahuluan

Chapter ini membangun **deep neural network** pertama (dengan hidden
layer) dan mempelajarinya dengan **backpropagation**. Contoh yang
dipakai adalah **masalah lampu lalu lintas (streetlight problem)**:
menentukan apakah aman menyeberang berdasarkan pola lampu.

Hal yang dibahas: data berupa matriks, mempelajari seluruh dataset,
full/batch/stochastic gradient descent, neural network mempelajari
**korelasi**, overfitting, tekanan naik dan turun (*up and down pressure*),
korelasi tidak langsung, serta nonlinearitas (ReLU) yang membuat layer
tambahan bermakna.

# Creating a Matrix or Two in Python

**Data lampu lalu lintas** disimpan sebagai matriks `streetlights` (tiap
baris = satu keadaan 3 lampu) dan `walk_vs_stop` (target: jalan atau
berhenti). NumPy menyediakan `np.array` dan operasi matriks untuk
memprosesnya.

In [ ]:
import numpy as np
weights = np.array([0.5,0.48,-0.7])
alpha = 0.1

streetlights = np.array( [ [ 1, 0, 1 ],
                           [ 0, 1, 1 ],
                           [ 0, 0, 1 ],
                           [ 1, 1, 1 ],
                           [ 0, 1, 1 ],
                           [ 1, 0, 1 ] ] )

walk_vs_stop = np.array( [ 0, 1, 0, 1, 1, 0 ] )

input = streetlights[0] # [1,0,1]
goal_prediction = walk_vs_stop[0] # equals 0... i.e. "stop"

for iteration in range(20):
    prediction = input.dot(weights)
    error = (goal_prediction - prediction) ** 2
    delta = prediction - goal_prediction
    weights = weights - (alpha * (input * delta))	

    print("Error:" + str(error) + " Prediction:" + str(prediction))

## Building Our Neural Network

Jaringan satu layer yang mempelajari **satu** baris data: prediksi,
hitung error, hitung `delta`, `weight_delta`, lalu perbarui weights.
Setelah banyak iterasi error mendekati nol pada baris tersebut.

In [ ]:
import numpy as np

a = np.array([0,1,2,1])
b = np.array([2,2,2,3])

print(a*b) #elementwise multiplication
print(a+b) #elementwise addition
print(a * 0.5) # vector-scalar multiplication
print(a + 0.5) # vector-scalar addition

# Learning the whole dataset!

**Mempelajari seluruh dataset:** loop melewati semua baris (*stochastic
gradient descent*, memperbarui weights per sampel). Alternatifnya *full*
(satu pembaruan untuk seluruh dataset) dan *batch* (kelompok sampel).
Jaringan bisa menyesuaikan weights agar benar pada semua baris, karena
ia belajar **korelasi** antara input dan target: lampu mana yang
berkorelasi dengan "jalan".

In [ ]:
import numpy as np

weights = np.array([0.5,0.48,-0.7])
alpha = 0.1

streetlights = np.array( [[ 1, 0, 1 ],
                          [ 0, 1, 1 ],
                          [ 0, 0, 1 ],
                          [ 1, 1, 1 ],
                          [ 0, 1, 1 ],
                          [ 1, 0, 1 ] ] )

walk_vs_stop = np.array( [ 0, 1, 0, 1, 1, 0 ] )

input = streetlights[0] # [1,0,1]
goal_prediction = walk_vs_stop[0] # equals 0... i.e. "stop"

for iteration in range(40):
    error_for_all_lights = 0
    for row_index in range(len(walk_vs_stop)):
        input = streetlights[row_index]
        goal_prediction = walk_vs_stop[row_index]
        
        prediction = input.dot(weights)
        
        error = (goal_prediction - prediction) ** 2
        error_for_all_lights += error
        
        delta = prediction - goal_prediction
        weights = weights - (alpha * (input * delta))	
        print("Prediction:" + str(prediction))
    print("Error:" + str(error_for_all_lights) + "\n")

# Our First "Deep" Neural Network

Pada pola tertentu, korelasi langsung antara input dan target tidak ada,
sehingga jaringan satu layer gagal. Solusinya: tambahkan **hidden layer**
yang membuat korelasi *perantara* (**creating correlation**), sehingga
layer berikutnya bisa mempelajari target. Layer pertama menghasilkan
representasi bantu, dan layer kedua memetakan representasi itu ke
target.

Agar layer tambahan bermakna, diperlukan **nonlinearitas**: tanpa itu dua
layer setara dengan satu layer linear. Buku memakai **ReLU**
(`relu(x) = max(0, x)`), yang membuat hidden node hanya "menyala" saat
positif (*sometimes correlation*).

In [ ]:
import numpy as np

np.random.seed(1)

def relu(x):
    return (x > 0) * x 

alpha = 0.2
hidden_size = 4

streetlights = np.array( [[ 1, 0, 1 ],
                          [ 0, 1, 1 ],
                          [ 0, 0, 1 ],
                          [ 1, 1, 1 ] ] )

walk_vs_stop = np.array([[ 1, 1, 0, 0]]).T

weights_0_1 = 2*np.random.random((3,hidden_size)) - 1
weights_1_2 = 2*np.random.random((hidden_size,1)) - 1

layer_0 = streetlights[0]
layer_1 = relu(np.dot(layer_0,weights_0_1))
layer_2 = np.dot(layer_1,weights_1_2)

# Backpropagation in Code

**Backpropagation:** menghitung `delta` output (`layer_2_delta = goal -
layer_2`), lalu mempropagasikan error **mundur** ke layer sebelumnya
dengan mengalikan delta dengan weights yang menghubungkan
(`layer_1_delta = layer_2_delta.dot(weights_1_2.T) * relu2deriv(layer_1)`).
Suku `relu2deriv` memastikan node yang tidak aktif tidak menyumbang
koreksi. Setiap weight lalu diperbarui dengan `weight_delta = layer.T.dot(delta)`.
Ini adalah *long-distance error attribution*: menentukan bagian error
yang menjadi tanggung jawab tiap weight yang jauh dari output.

In [ ]:
import numpy as np

np.random.seed(1)

def relu(x):
    return (x > 0) * x # returns x if x > 0
                       # return 0 otherwise

def relu2deriv(output):
    return output>0 # returns 1 for input > 0
                    # return 0 otherwise
alpha = 0.2
hidden_size = 4

weights_0_1 = 2*np.random.random((3,hidden_size)) - 1
weights_1_2 = 2*np.random.random((hidden_size,1)) - 1

for iteration in range(60):
   layer_2_error = 0
   for i in range(len(streetlights)):
      layer_0 = streetlights[i:i+1]
      layer_1 = relu(np.dot(layer_0,weights_0_1))
      layer_2 = np.dot(layer_1,weights_1_2)

      layer_2_error += np.sum((layer_2 - walk_vs_stop[i:i+1]) ** 2)

      layer_2_delta = (walk_vs_stop[i:i+1] - layer_2)
      layer_1_delta=layer_2_delta.dot(weights_1_2.T)*relu2deriv(layer_1)

      weights_1_2 += alpha * layer_1.T.dot(layer_2_delta)
      weights_0_1 += alpha * layer_0.T.dot(layer_1_delta)

   if(iteration % 10 == 9):
      print("Error:" + str(layer_2_error))

# One Iteration of Backpropagation

Satu iterasi backpropagation secara lengkap: forward pass untuk
mendapatkan `layer_1` dan `layer_2`, hitung delta dari output ke hidden,
lalu hitung `weight_delta` untuk kedua matriks weights dan perbarui.

In [ ]:
import numpy as np

np.random.seed(1)

def relu(x):
    return (x > 0) * x 

def relu2deriv(output):
    return output>0 

lights = np.array( [[ 1, 0, 1 ],
                    [ 0, 1, 1 ],
                    [ 0, 0, 1 ],
                    [ 1, 1, 1 ] ] )

walk_stop = np.array([[ 1, 1, 0, 0]]).T

alpha = 0.2
hidden_size = 3

weights_0_1 = 2*np.random.random((3,hidden_size)) - 1
weights_1_2 = 2*np.random.random((hidden_size,1)) - 1

layer_0 = lights[0:1]
layer_1 = np.dot(layer_0,weights_0_1)
layer_1 = relu(layer_1)
layer_2 = np.dot(layer_1,weights_1_2)

error = (layer_2-walk_stop[0:1])**2

layer_2_delta=(layer_2-walk_stop[0:1])

layer_1_delta=layer_2_delta.dot(weights_1_2.T)
layer_1_delta *= relu2deriv(layer_1)

weight_delta_1_2 = layer_1.T.dot(layer_2_delta)
weight_delta_0_1 = layer_0.T.dot(layer_1_delta)

weights_1_2 -= alpha * weight_delta_1_2
weights_0_1 -= alpha * weight_delta_0_1

## Putting it all Together

Kode lengkap jaringan 3 layer yang mempelajari dataset lampu lalu lintas
dengan backpropagation. Error turun mendekati nol: jaringan *deep*
pertama berhasil mempelajari pola yang tidak bisa dipelajari jaringan
satu layer.

In [ ]:
import numpy as np

np.random.seed(1)

def relu(x):
    return (x > 0) * x # returns x if x > 0
                       # return 0 otherwise

def relu2deriv(output):
    return output>0 # returns 1 for input > 0
                    # return 0 otherwise

streetlights = np.array( [[ 1, 0, 1 ],
                          [ 0, 1, 1 ],
                          [ 0, 0, 1 ],
                          [ 1, 1, 1 ] ] )

walk_vs_stop = np.array([[ 1, 1, 0, 0]]).T
    
alpha = 0.2
hidden_size = 4

weights_0_1 = 2*np.random.random((3,hidden_size)) - 1
weights_1_2 = 2*np.random.random((hidden_size,1)) - 1

for iteration in range(60):
   layer_2_error = 0
   for i in range(len(streetlights)):
      layer_0 = streetlights[i:i+1]
      layer_1 = relu(np.dot(layer_0,weights_0_1))
      layer_2 = np.dot(layer_1,weights_1_2)

      layer_2_error += np.sum((layer_2 - walk_vs_stop[i:i+1]) ** 2)

      layer_2_delta = (layer_2 - walk_vs_stop[i:i+1])
      layer_1_delta=layer_2_delta.dot(weights_1_2.T)*relu2deriv(layer_1)

      weights_1_2 -= alpha * layer_1.T.dot(layer_2_delta)
      weights_0_1 -= alpha * layer_0.T.dot(layer_1_delta)

   if(iteration % 10 == 9):
      print("Error:" + str(layer_2_error))

## Key Ideas

- Neural network belajar korelasi antara input dan output.
- Hidden layer membuat korelasi perantara bagi layer berikutnya.
- Tanpa nonlinearitas, banyak layer setara dengan satu layer.
- Backpropagation mempropagasikan delta mundur untuk memperbarui weights tiap layer.
- ReLU adalah nonlinearitas sederhana: max(0, x).

## Kesimpulan


Chapter 6 membangun jaringan *deep* pertama dan memperkenalkan
**backpropagation**: error dari output dipropagasikan mundur sehingga
tiap layer mendapat delta untuk memperbarui weights-nya. Hal-hal penting:

- Jaringan belajar **korelasi**; tanpa korelasi langsung, jaringan satu
  layer gagal.
- **Hidden layer** menciptakan korelasi perantara.
- **Nonlinearitas** (ReLU) diperlukan agar layer tambahan tidak runtuh
  menjadi satu layer linear.
- Backpropagation memungkinkan gradient descent dipakai pada jaringan
  yang dalam.